In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from xgboost import XGBClassifier
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import joblib

In [ ]:
df = pd.read_csv('../DataSet/ml_ready_data.csv')

In [ ]:
df.drop(columns=['Seniority'], inplace=True)

In [ ]:
df

In [ ]:
df['AI_Impact_Level'].value_counts()

In [ ]:
plt.figure(figsize=(18,14))

sns.heatmap(
df.corr(),
annot=True,
fmt=".2f",
cmap="Reds",
linewidths=0.5,
annot_kws={"size":8}
)

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
suspicious_columns = [
    'Is_High_Risk', 
    'Automation_Risk', 
    'Risk_Category', 
    'Automation_Probability_2030', 
    'AI_Exposure_Index',
]

features = df.drop(columns=['AI_Impact_Level'] + suspicious_columns, errors='ignore')
target = df['AI_Impact_Level']

X_train, X_test, y_train, y_test = train_test_split(features ,target , test_size=0.2, random_state=42)

## Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
acc_lr

## Random Forest Model

In [ ]:
estimators_to_test = [50, 75, 100, 250, 300 ,400 ,500]

In [ ]:
for n in estimators_to_test:
    rf_model = RandomForestClassifier(
        n_estimators=n, 
        max_depth=4, 
        random_state=42, 
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    acc

In [ ]:
rf_model = RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1 ,max_depth=4)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
accuracy

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:

importances = rf_model.feature_importances_

feature_names = X_train.columns
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})
importance_df.drop(index=importance_df[importance_df['Feature'] == 'Projected_Openings_(2030)'].index, inplace=True) 
importance_df.drop(index=importance_df[importance_df['Feature'] == 'Job_Openings_(2024)'].index, inplace=True) 

importance_df = importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(10), x='Importance', y='Feature', palette='magma')
plt.title('Top 10 Drivers of AI Impact', fontsize=14, fontweight='bold')
plt.xlabel('Importance (Percentage of Decision Making)')
plt.ylabel('Job Feature')
plt.tight_layout()
plt.show()

## XGBoost Classifier

In [ ]:
X_train

In [ ]:
for n in estimators_to_test:
    xgb_boosting_model = XGBClassifier(n_estimators=n, max_depth=4, random_state=42, use_label_encoder=False, eval_metric='mlogloss')
    xgb_boosting_model.fit(X_train, y_train)
    y_pred_xgb = xgb_boosting_model.predict(X_test)
    acc_xgb = accuracy_score(y_test, y_pred_xgb)
    print(f"XGBoosting Model with {n} Estimators: {acc_xgb}%")

In [ ]:
xgb_boosting_model = XGBClassifier(n_estimators=250, max_depth=4, random_state=42, eval_metric='mlogloss')
xgb_boosting_model.fit(X_train, y_train)
y_pred_xgb = xgb_boosting_model.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"XGBoosting Model Accuracy: {acc_xgb:.2f}%\n")

In [ ]:
cm = confusion_matrix(y_test, y_pred_xgb)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='magma', 
            xticklabels=['High', 'Low', 'Moderate'], 
            yticklabels=['High', 'Low', 'Moderate'])
plt.title('XGBoost Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
xgb_boosting_model = XGBClassifier(
    n_estimators=50,      
    max_depth=4,          
    learning_rate=0.05,
    random_state=42, 
    eval_metric='mlogloss'
)
xgb_boosting_model.fit(X_train, y_train)
y_pred_xgb_h = xgb_boosting_model.predict(X_test)
acc_xgb_h = accuracy_score(y_test, y_pred_xgb_h)
acc_xgb_h

In [ ]:
cm = confusion_matrix(y_test, y_pred_xgb_h)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='magma', 
            xticklabels=['High', 'Low', 'Moderate'], 
            yticklabels=['High', 'Low', 'Moderate'])
plt.title('XGBoost Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
print(f"Random Forest:{accuracy_score(y_test, y_pred):.2f}")
print(f"XGBoosting before hyperparameter tuning:{acc_xgb:.2f}")
print(f"XGBoosting after hyperparameter tuning:{acc_xgb_h:.2f}")
print(f"Logistic Regression:{acc_lr:.2f}")

In [ ]:
joblib.dump(xgb_boosting_model, '../DataSet/model/xgb_boosting_model.pkl')
print("Champion Model saved successfully for Streamlit!")